In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import display, Markdown, update_display
from scrapper import fetch_website_links_farhan, fetch_website_contents_farhan
from openai import OpenAI

In [2]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API Key? Please visit the troubleshooting notebook")

MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links_farhan("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

In [4]:
link_system_prompt = """
you are provided with a list of links on a webpage.
You are able to check which of these links would be most relevant to include in a brochure about the company,
such as links to an About Page, or a Company page, or Careers/Jobs pages.
You should respond in a JSON as in this example:
{
    "links":
    [
        {"type": "about page", "url":"https://full.url/goes/here/about"},
        {"type": "career page","url":"https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} - 
    Please decide which of these links are relevant web links for a brochure about the company,
    respond with the full https URL in json format.
    Do not include terms of Service, Privacy, email links.

    Links (some might be a relative links)
    """
    links = fetch_website_links_farhan(url)
    user_prompt+="\n".join(links)
    return user_prompt


In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


    Here is the list of links on the website https://edwarddonner.com - 
    Please decide which of these links are relevant web links for a brochure about the company,
    respond with the full https URL in json format.
    Do not include terms of Service, Privacy, email links.

    Links (some might be a relative links)

    #wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
http

In [8]:
def select_relevant_links(url):
    print(f"selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format = {"type":"json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links["links"])} relevant links")
    return links




In [9]:
select_relevant_links("https://huggingface.co")

selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


{'links': [{'type': 'company page', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'career page', 'url': 'https://apply.workable.com/huggingface/'}]}

In [10]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents_farhan(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page: \n\n{contents}\n## Relavant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents_farhan(link['url'])
    return result

In [11]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links
## Landing Page: 

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
8 days ago
•
1.62M
•
1.63k
zai-org/GLM-5.2
Updated
4 days ago
•
231k
•
3.52k
baidu/Unlimited-OCR
Updated
4 days ago
•
1.07M
•
1.79k
InternScience/Agents-A1
Updated
3 days ago
•
8.77k
•
337
tencent/Hy3
Updated
about 5 hours ago
•
2
•
291
Brow

In [12]:
brochure_system_prompt = """
You are a assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in Markdown without code blocks.
Include details of a company culture, customers and careers/jobs if you have the information.
"""

In [13]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
    You are looking at a company called {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code block. \n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [14]:
get_brochure_user_prompt(company_name="HuggingFace", url="https://huggingface.co")

selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


'\n    You are looking at a company called HuggingFace\n    Here are the contents of its landing page and other relevant pages;\n    use this information to build a short brochure of the company in markdown without code block. \n\n\n    ## Landing Page: \n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nempero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF\nUpdated\n8 days ago\n•\n1.62M\n•\n1.63k\nzai-org/GLM-5.2\nUpdate

In [15]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [16]:
create_brochure(company_name="HuggingFace", url="https://huggingface.co")

selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the premier AI community building the future of machine learning. Acting as a collaborative platform, it empowers the global machine learning community to create, discover, and share models, datasets, and applications freely and efficiently. Hugging Face serves as a centralized hub where researchers, developers, and enterprises converge to innovate and accelerate AI advancements.

---

## What Hugging Face Offers

- **Models:** Access and contribute to over **2 million machine learning models** spanning various AI tasks and languages. Models are regularly updated and cover diverse domains such as natural language processing, computer vision, speech, and more.

- **Datasets:** Explore and collaborate on **500,000+ datasets** publicly available for training and benchmarking machine learning models.

- **Spaces:** Build, deploy, and share interactive AI applications easily, fostering a space for real-time engagement and demonstrations.

- **Buckets and Storage:** Provide scalable solutions for storing, managing, and serving datasets and model artifacts efficiently.

- **HuggingChat:** A conversational AI platform embedded within the ecosystem to enable natural language interactions.

- **Enterprise Solutions:** Custom support and tools designed for business needs, including enterprise-grade support, inference providers, endpoints, and pro services.

---

## Community & Collaboration

Hugging Face thrives on a vibrant and inclusive community. It offers:

- A **dedicated forum** and an active **Discord server** for discussion, problem-solving, and networking.

- **Open source contributions** via their GitHub repositories, fostering transparency and shared growth in AI research.

- Regular updates via a **blog**, **daily research papers**, and educational resources to keep members informed and skilled.

Their collaboration platform is designed to let users host and work on unlimited public models, datasets, and apps, thereby creating a dynamic ecosystem for AI advancement.

---

## Company Culture

Hugging Face is built on openness, inclusivity, and community-driven innovation. The company values:

- **Collaboration** — fostering teamwork across the globe to accelerate AI progress.

- **Transparency** — maintaining open-source principles and accessible knowledge sharing.

- **Innovation** — pushing the boundaries of machine learning through continual research and infrastructure development.

- **Empowerment** — equipping developers, researchers, and enterprises with easy-to-use AI tools and scalable resources.

---

## Careers at Hugging Face

Hugging Face offers exciting career opportunities for passionate AI practitioners who want to be part of a vibrant community shaping the future of machine learning technologies. The environment encourages continuous learning, collaboration, and contributing to open source projects on a global scale.

Prospective recruits can expect to work alongside industry leaders, with exposure to cutting-edge ML models, large-scale datasets, and innovative AI applications.

To explore openings and join the Hugging Face team, visit their career pages and engage directly with their community forums.

---

## Customers & Users

Hugging Face serves a broad range of users including:

- **Researchers** seeking state-of-the-art models and datasets to accelerate AI experiments.

- **Developers** building applications powered by machine learning.

- **Enterprises** looking for scalable, reliable AI solutions with dedicated support.

- **Educators and Students** utilizing the platform as a learning resource and collaboration tool.

- **Open Source Contributors** who extend and refine models and tools that the entire community benefits from.

---

## Brand & Visual Identity

- The Hugging Face brand is represented by friendly, accessible visual assets: warm colors (#FFD21E, #FF9D00, #6B7280) that reflect optimism and energy.

- Official brand resources including logos in multiple formats (.svg, .png, .ai) are publicly available to promote consistent and trusted brand use.

---

## Connect & Learn More

- Website: [huggingface.co](https://huggingface.co)  
- Community Forum and Discord for active discussions  
- GitHub Repositories for source code and collaboration  
- Blog and Daily Papers for news and research updates  

Join the AI community that's shaping tomorrow — innovate, collaborate, and accelerate AI development with Hugging Face.

---

*Hugging Face: The AI Community Building the Future.*

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],stream = True
    )
    
    response = ""
    display_handle = display(Markdown(""),display_id=True)
    for chunk in stream:
        response+= chunk.choices[0].delta.content or ''
        update_display(Markdown(response),display_id=display_handle.display_id)

In [ ]:
stream_brochure(company_name="HuggingFace", url="https://huggingface.co")